In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from typing import Dict, List, Set, Tuple
from scipy import stats
import pysam

In [ ]:
## Set paths for test locus tebp-beta
LOCUS_NAME = 'tebp_beta'

## TSV exported from assembly notebook. Required columns: sequence, score
PATH_SCORES_TSV = '/path/to/TEBP_beta_sorted_paths.tsv'

## FASTQs used to build the assembly
TRAINING_FASTQS = [
    '/path/to/tebp_beta_36h_candidate_reads.fastq',
    '/path/to/tebp_beta_48h_candidate_reads.fastq',
]

## Independent test BAMs
TEST_BAMS = {
    '36h': '/path/to/candidate_tebp_beta_intermediates_36h.bam',
    '48h': '/path/to/candidate_tebp_beta_intermediates_36h.bam',
    '60h': '/path/to/candidate_tebp_beta_intermediates_36h.bam',
}

## directory for output figures
FIGURE_DIR = '.'   # default: same directory as this notebook

## coverage parameters
K = 25   # k-mer size
COVERAGE_THRESHOLD = 0.5  # a read "supports" a path if it covers >= this fraction of path k-mers

In [ ]:
## helpers
DNA = set('ATCG')

def clean_seq(seq: str) -> str:
    return ''.join(c for c in seq.upper() if c in DNA)

def rc(seq: str) -> str:
    return seq.translate(str.maketrans('ATCGatcg', 'TAGCtagc'))[::-1].upper()

def kmer_set(seq: str, k: int) -> Set[str]:
    seq = clean_seq(seq)
    return {min(seq[i:i+k], rc(seq[i:i+k])) for i in range(len(seq) - k + 1)
            if set(seq[i:i+k]) <= DNA}

def load_path_scores(tsv_path: str) -> pd.DataFrame:
    """Load assembly-output TSV (columns: sequence, score). Sorted by score"""
    df = pd.read_csv(tsv_path, sep='\t')
    if 'sequence' not in df.columns or 'score' not in df.columns:
        raise ValueError(f'TSV must have columns sequence, score; got {df.columns.tolist()}')
    df = df.sort_values('score', ascending=False).reset_index(drop=True)
    df['path_idx'] = df.index
    df['length']   = df['sequence'].str.len()
    return df

def load_reads_from_bam(bam_path: str) -> List[Tuple[str, str]]:
    best = {}
    n_records = n_no_seq = 0
    with pysam.AlignmentFile(bam_path, 'rb') as bam:
        for r in bam:
            if r.is_unmapped:
                continue
            n_records += 1
            seq = r.query_sequence
            if not seq:
                n_no_seq += 1
                continue
            priority = 2 if r.is_secondary else (1 if r.is_supplementary else 0)
            cur = best.get(r.query_name)
            if cur is None or priority < cur[0] or (priority == cur[0] and len(seq) > len(cur[1])):
                best[r.query_name] = (priority, seq)
    print(f'  {bam_path.split("/")[-1]}: {n_records} records → {len(best)} unique reads')
    return [(name, seq) for name, (_, seq) in best.items()]

def load_fastq_names(fastq_path: str) -> Set[str]:
    """Return set of read names from a FASTQ file."""
    names = set()
    with open(fastq_path) as f:
        for i, line in enumerate(f):
            if i % 4 == 0 and line.startswith('@'):
                names.add(line[1:].split()[0])
    return names

In [ ]:
## load path scores
paths_df  = load_path_scores(PATH_SCORES_TSV)
path_kmers = [kmer_set(s, K) for s in paths_df['sequence']]
print(f'Loaded {len(paths_df)} paths  '
      f'(lengths: min={paths_df["length"].min()}, '
      f'median={int(paths_df["length"].median())}, '
      f'max={paths_df["length"].max()})')

## Load training read names for leakage detection
training_names: Set[str] = set()
for fq in TRAINING_FASTQS:
    n_before = len(training_names)
    training_names |= load_fastq_names(fq)
    print(f'  training FASTQ {os.path.basename(fq)}: {len(training_names)-n_before} names')
print(f'Total training read names: {len(training_names)}')

# Load test reads per timepoint
timepoints: Dict[str, dict] = {}
for tp, bam_path in TEST_BAMS.items():
    reads      = load_reads_from_bam(bam_path)
    is_train   = [name in training_names for name, _ in reads]
    rk_all     = [kmer_set(s, K) for _, s in reads]
    idx_indep  = [i for i, t in enumerate(is_train) if not t]
    idx_train  = [i for i, t in enumerate(is_train) if t]
    timepoints[tp] = {
        'reads':           reads,
        'read_kmers':      rk_all,
        'is_training':     is_train,
        'idx_indep':       idx_indep,
        'idx_training':    idx_train,
        'read_kmers_indep': [rk_all[i] for i in idx_indep],
    }
    print(f'  [{tp}] {len(reads)} reads = {len(idx_train)} training (leaked) + {len(idx_indep)} independent')

In [ ]:
## per-path coverage
# For each candidate path, computes:
# pooled coverage: fraction of path k-mers present in the union of all independent reads' k-mer sets
# n_supp: number of individual reads that each cover ≥ `COVERAGE_THRESHOLD` (0.5) of the path's k-mers

def path_metrics(path_kmers: Set[str], read_kmer_sets: List[Set[str]],
                 threshold: float = COVERAGE_THRESHOLD) -> dict:
    if not path_kmers or not read_kmer_sets:
        return {'n_supp': 0, 'best_single_read_cov': 0.0, 'pooled_cov': 0.0}
    n_path   = len(path_kmers)
    n_supp   = 0
    best_cov = 0.0
    union_rk: Set[str] = set()
    for rk in read_kmer_sets:
        cov = len(path_kmers & rk) / n_path
        if cov >= threshold:
            n_supp += 1
        if cov > best_cov:
            best_cov = cov
        union_rk |= rk
    return {
        'n_supp':               n_supp,
        'best_single_read_cov': best_cov,
        'pooled_cov':           len(path_kmers & union_rk) / n_path,
    }

rows = []
for path_idx, pk in enumerate(path_kmers):
    row = {'path_idx': path_idx}
    all_rk, indep_rk = [], []
    for tp, td in timepoints.items():
        m_all   = path_metrics(pk, td['read_kmers'])
        m_indep = path_metrics(pk, td['read_kmers_indep'])
        row[f'n_supp_{tp}_indep']     = m_indep['n_supp']
        row[f'pooled_cov_{tp}_indep'] = m_indep['pooled_cov']
        row[f'n_supp_{tp}_all']       = m_all['n_supp']
        row[f'pooled_cov_{tp}_all']   = m_all['pooled_cov']
        all_rk.extend(td['read_kmers'])
        indep_rk.extend(td['read_kmers_indep'])
    m_tot_indep = path_metrics(pk, indep_rk)
    m_tot_all   = path_metrics(pk, all_rk)
    row['n_supp_total_indep']     = m_tot_indep['n_supp']
    row['pooled_cov_total_indep'] = m_tot_indep['pooled_cov']
    row['n_supp_total_all']       = m_tot_all['n_supp']
    row['pooled_cov_total_all']   = m_tot_all['pooled_cov']
    rows.append(row)

per_path = paths_df.merge(pd.DataFrame(rows), on='path_idx')

n_paths     = len(per_path)
n_supported = int((per_path['pooled_cov_total_indep'] >= 0.5).sum())
med_cov     = per_path['pooled_cov_total_indep'].median()
n_single    = int((per_path['n_supp_total_indep'] >= 1).sum())
print(f'Paths with pooled cov ≥ 0.5 (independent, pooled timepoints): {n_supported}/{n_paths} ({n_supported/n_paths:.1%})')
print(f'Median pooled coverage: {med_cov:.2f}')
print(f'Paths with ≥1 single-read supporter:                           {n_single}/{n_paths} ({n_single/n_paths:.1%})')

In [ ]:
## calculating per-read best path coverage
# For each independent test read, the best per-read coverage is the fraction of that read's k-mers
# present in the closest-matching predicted path

per_read = {}
for tp, td in timepoints.items():
    rows_r = []
    for (rid, _), rk, is_train in zip(td['reads'], td['read_kmers'], td['is_training']):
        if not rk:
            continue
        n_read   = len(rk)
        best_cov = max((len(rk & pk) / n_read for pk in path_kmers if pk), default=0.0)
        rows_r.append({'read_id': rid, 'best_coverage': best_cov, 'is_training': is_train})
    per_read[tp] = pd.DataFrame(rows_r)

for tp, df in per_read.items():
    sub = df[~df['is_training']]
    if sub.empty:
        continue
    print(f'[{tp}]  n_indep={len(sub)}  mean={sub["best_coverage"].mean():.2f}  '
          f'frac≥0.8={( sub["best_coverage"]>=0.8).mean():.1%}')

In [ ]:
## plotting figure 1d
# Use timepoints with ≥5 independent reads; add pooled panel
panel_tps = [tp for tp in TPS if len(timepoints[tp]['idx_indep']) >= 5]
pooled_df = pd.concat([per_read[tp][~per_read[tp]['is_training']] for tp in TPS
                        if tp in per_read], ignore_index=True)
panels = [(tp, per_read[tp][~per_read[tp]['is_training']], COLORS[tp]) for tp in panel_tps]
panels.append(('pooled', pooled_df, DARK))

fig2, axes2 = plt.subplots(1, len(panels), figsize=(4.5*len(panels), 4.0), sharey=True)
if len(panels) == 1:
    axes2 = [axes2]
bins = np.linspace(0, 1, 21)
for ax, (label, sub, col) in zip(axes2, panels):
    ax.hist(sub['best_coverage'], bins=bins, color=col, edgecolor='white', alpha=0.9)
    mean_c = sub['best_coverage'].mean()
    ax.axvline(mean_c, color='crimson', ls='--', lw=1.4, label=f'mean={mean_c:.2f}')
    ax.axvline(0.8,    color='gray',    ls=':',  lw=1.0, label='0.8 threshold')
    pct = (sub['best_coverage'] >= 0.8).mean() * 100
    ax.set_title(f'{label}  (n={len(sub)})\n{pct:.0f}% reads ≥ 0.8', fontsize=12)
    ax.set_xlabel('Best per-read coverage', fontsize=11)
    ax.legend(fontsize=9, frameon=False)
axes2[0].set_ylabel('# reads', fontsize=11)
fig2.suptitle(f'{LOCUS_NAME} — per-read best-coverage (independent reads only)', fontsize=13, y=1.02)
plt.tight_layout()
out2 = os.path.join(FIGURE_DIR, f'{LOCUS_NAME}_fig1d_per_read_coverage.png')
fig2.savefig(out2, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## bayesian score validation
#  Overall Spearman ρ and Within-length-quartile Spearman between raw score and independent read support
    
per_path['score_per_bp'] = per_path['score'] / per_path['length']
target = per_path['n_supp_total_indep']

rho_raw,  p_raw  = stats.spearmanr(per_path['score'],        target)
rho_norm, p_norm = stats.spearmanr(per_path['score_per_bp'], target)

n        = len(per_path)
n_dec    = max(1, n // 10)
top_med  = per_path.nlargest(n_dec,  'score')[target.name].median()
bot_med  = per_path.nsmallest(n_dec, 'score')[target.name].median()

print(f'Raw score   Spearman ρ = {rho_raw:+.3f}  p = {p_raw:.4f}')
print(f'Score/bp    Spearman ρ = {rho_norm:+.3f}  p = {p_norm:.4f}')
print(f'Top decile median indep reads: {top_med}  |  Bottom decile: {bot_med}')

# Within-quartile Spearman
per_path['len_q'] = pd.qcut(per_path['length'], q=4,
                              labels=['Q1 (short)','Q2','Q3','Q4 (long)'], duplicates='drop')
print()
for q, sub in per_path.groupby('len_q', observed=True):
    if len(sub) >= 3 and sub['score'].nunique() > 1 and sub['n_supp_total_indep'].nunique() > 1:
        rho_q, p_q = stats.spearmanr(sub['score'], sub['n_supp_total_indep'])
        sig = ' *' if p_q < 0.05 else ''
        print(f'  {str(q):<12}  n={len(sub)}  len {int(sub["length"].min())}–{int(sub["length"].max())} bp  '
              f'ρ={rho_q:+.3f}  p={p_q:.3f}{sig}')
    else:
        print(f'  {str(q):<12}  n={len(sub)}  (insufficient variation for Spearman)')

In [ ]:
## plotting figure 1e and 1f

q_labels, q_rhos, q_ps = [], [], []
for q, sub in per_path.groupby('len_q', observed=True):
    q_labels.append(str(q))
    if len(sub) >= 3 and sub['score'].nunique() > 1 and sub['n_supp_total_indep'].nunique() > 1:
        rho_q, p_q = stats.spearmanr(sub['score'], sub['n_supp_total_indep'])
    else:
        rho_q, p_q = float('nan'), float('nan')
    q_rhos.append(rho_q)
    q_ps.append(p_q)

fig3, (axA, axC) = plt.subplots(1, 2, figsize=(11, 4.3))

# Panel A: score vs independent read support
sc = axA.scatter(per_path['score'], per_path['n_supp_total_indep'],
                 c=per_path['length'], cmap='viridis_r', s=55, alpha=0.85, edgecolors='none')
plt.colorbar(sc, ax=axA, label='Path length (bp)', shrink=0.85)
axA.set_xlabel('Bayesian path score', fontsize=11)
axA.set_ylabel('# supporting independent reads', fontsize=11)
axA.set_title(f'Score tracks read support\n(Spearman ρ = {rho_raw:+.3f}, p = {p_raw:.1e})', fontsize=11)

# Panel C: within-quartile Spearman bars
bar_c = ['steelblue' if not np.isnan(r) else 'lightgray' for r in q_rhos]
bars  = axC.bar(q_labels, q_rhos, color=bar_c, edgecolor='white', width=0.55)
for bar, rho, p in zip(bars, q_rhos, q_ps):
    if not np.isnan(rho):
        sig = ' *' if p < 0.05 else ''
        axC.text(bar.get_x()+bar.get_width()/2, rho+0.03,
                 f'{rho:+.2f}{sig}', ha='center', va='bottom', fontsize=10)
axC.axhline(0, color='black', lw=0.8)
axC.set_ylim(-0.3, 1.2)
axC.set_xlabel('Length quartile', fontsize=11)
axC.set_ylabel('Spearman ρ (score vs. indep. reads)', fontsize=11)
axC.set_title('Within-length-stratum signal\n(* p < 0.05)', fontsize=11)

for a in (axA, axC):
    a.spines['top'].set_visible(False)
    a.spines['right'].set_visible(False)

fig3.suptitle(f'{LOCUS_NAME} — Bayesian scoring validation', fontsize=12, y=1.02)
plt.tight_layout()
out3 = os.path.join(FIGURE_DIR, f'{LOCUS_NAME}_fig1ef_scoring_validation.png')
fig3.savefig(out3, dpi=150, bbox_inches='tight')
plt.show()